# Day 6：Alpaca 指令微调实践 — 完整 SFT 实验

> **目标**：在小规模数据上完成一次完整的 Alpaca 风格指令微调实验——从数据处理、Prompt Template 构造、Loss Mask 实现，到 SFT 训练循环、Loss 曲线监控，最终进行文本生成评估；体验从「预训练补全模型」到「指令遵循模型」的行为转变。

---

## 总体流程

```
Part 1: 数据准备
  加载指令数据 → Prompt Template 填充 → Tokenize → Loss Mask

Part 2: 模型构建
  使用 Week4 手写的 LLaMA 模型（缩小配置）

Part 3: SFT 训练
  Loss Mask 训练循环 → Loss 曲线监控 → 检查点保存

Part 4: 生成与评估
  对比训练前后的模型行为 → 指令遵循效果评估

Part 5: Adapter 微调对比（可选）
  在同一模型上对比 Full FT vs Adapter
```

In [ ]:
import json
import math
import os
import time
import copy
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")

## Part 1：数据准备

### 1.1 构造指令微调数据

我们构造一组覆盖多种任务类型的 SFT 数据。格式遵循 Stanford Alpaca 的标准三元组：`(instruction, input, output)`。

在真实场景中，这些数据可以来自：
- Day 3 手写的 Self-Instruct 管线
- LLM API 批量生成
- 人工标注

In [ ]:
SFT_DATA = [
    # ====== 问答 ======
    {
        "instruction": "What is machine learning?",
        "input": "",
        "output": "Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed. It uses algorithms to build mathematical models from training data, allowing systems to make predictions or decisions."
    },
    {
        "instruction": "Explain the concept in simple terms.",
        "input": "Neural networks are computational systems inspired by biological neural networks that constitute animal brains.",
        "output": "A neural network is like a digital brain. It has many small units connected together, similar to neurons in your brain. These units learn to recognize patterns by adjusting their connections based on examples, just like how you learn from experience."
    },
    {
        "instruction": "What is the capital of France?",
        "input": "",
        "output": "The capital of France is Paris. It is located in northern France along the Seine River and is the country's largest city."
    },
    {
        "instruction": "Who wrote Romeo and Juliet?",
        "input": "",
        "output": "Romeo and Juliet was written by William Shakespeare. It is a tragedy written around 1594 to 1596 and is one of his most famous plays."
    },

    # ====== 文本生成 ======
    {
        "instruction": "Write a short poem about the moon.",
        "input": "",
        "output": "Silver light upon the hill,\nThe moon hangs bright and still.\nStars surround her gentle glow,\nWatching over the world below."
    },
    {
        "instruction": "Write a professional email.",
        "input": "Decline a meeting scheduled for Monday at 3pm",
        "output": "Subject: Unable to Attend Monday Meeting\n\nDear Team,\n\nI apologize, but I will not be able to attend the meeting scheduled for Monday at 3pm due to a prior commitment. Could we reschedule, or would someone be able to share the meeting notes?\n\nBest regards"
    },
    {
        "instruction": "Create a brief story opening.",
        "input": "Genre: mystery, Setting: old library",
        "output": "The dust-covered books hadn't been touched in decades. But tonight, under the flickering lamplight, Sarah noticed something peculiar: one book on the top shelf was completely free of dust, as if someone had been reading it very recently."
    },

    # ====== 分类 ======
    {
        "instruction": "Classify the sentiment of the following text as positive, negative, or neutral.",
        "input": "This restaurant has amazing food but the service was terribly slow.",
        "output": "Mixed (leaning negative). While the food quality is praised, the overall experience is negatively impacted by the poor service speed."
    },
    {
        "instruction": "Determine whether this statement is a fact or opinion.",
        "input": "Python is the best programming language.",
        "output": "Opinion. While Python is widely popular and versatile, calling it 'the best' is subjective. Different languages excel in different domains."
    },

    # ====== 改写与摘要 ======
    {
        "instruction": "Summarize the following text in one sentence.",
        "input": "The transformer architecture was introduced in 2017 and revolutionized natural language processing. It replaced recurrent networks with self-attention mechanisms, allowing better parallelization during training.",
        "output": "The transformer architecture (2017) replaced recurrent networks with self-attention, enabling better parallelization and revolutionizing NLP."
    },
    {
        "instruction": "Rewrite the sentence in a more formal tone.",
        "input": "The project's gonna be done by next week, no worries.",
        "output": "The project is expected to be completed by the end of next week. Please rest assured that progress is on track."
    },

    # ====== 推理 ======
    {
        "instruction": "Solve the following math problem step by step.",
        "input": "A store has 45 apples. They sell 18 in the morning and receive 12 more in the afternoon. How many apples do they have?",
        "output": "Step 1: Start with 45 apples.\nStep 2: Sell 18 apples: 45 - 18 = 27 apples.\nStep 3: Receive 12 more: 27 + 12 = 39 apples.\nThe store has 39 apples."
    },
    {
        "instruction": "What comes next in the sequence?",
        "input": "2, 6, 18, 54, ?",
        "output": "The next number is 162. This is a geometric sequence where each term is multiplied by 3: 2x3=6, 6x3=18, 18x3=54, 54x3=162."
    },

    # ====== 代码 ======
    {
        "instruction": "Write a Python function to check if a number is prime.",
        "input": "",
        "output": "def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True"
    },
    {
        "instruction": "Explain what this code does.",
        "input": "[x**2 for x in range(10) if x % 2 == 0]",
        "output": "This is a list comprehension that generates squares of even numbers from 0 to 9. It iterates through 0-9, filters for even numbers (x%2==0), and squares each: [0, 4, 16, 36, 64]."
    },

    # ====== 翻译 ======
    {
        "instruction": "Translate the following to French.",
        "input": "Good morning, how are you today?",
        "output": "Bonjour, comment allez-vous aujourd'hui?"
    },

    # ====== 信息提取 ======
    {
        "instruction": "Extract all numbers mentioned in the text.",
        "input": "The company was founded in 2015 with 5 employees. By 2023, it had grown to 250 employees with revenue of 10 million dollars.",
        "output": "Numbers found: 2015 (founding year), 5 (initial employees), 2023 (reference year), 250 (current employees), 10 million (revenue in dollars)."
    },

    # ====== 更多问答以增加数据量 ======
    {
        "instruction": "What are the three states of matter?",
        "input": "",
        "output": "The three common states of matter are solid, liquid, and gas. In a solid, particles are tightly packed. In a liquid, particles are close but can move. In a gas, particles are far apart and move freely."
    },
    {
        "instruction": "Give three tips for staying healthy.",
        "input": "",
        "output": "1. Exercise regularly - aim for at least 30 minutes of moderate activity daily.\n2. Eat a balanced diet rich in fruits, vegetables, and whole grains.\n3. Get adequate sleep - most adults need 7 to 9 hours per night."
    },
    {
        "instruction": "List the advantages and disadvantages.",
        "input": "Remote work",
        "output": "Advantages: flexible schedule, no commute time, better work-life balance, access to global talent pool.\nDisadvantages: isolation and loneliness, difficulty separating work and personal life, communication challenges, potential for overworking."
    },
]

print(f"SFT 数据量: {len(SFT_DATA)} 条")
has_input = sum(1 for d in SFT_DATA if d['input'].strip())
print(f"有 input: {has_input} 条, 无 input: {len(SFT_DATA) - has_input} 条")

### 1.2 Prompt Template

使用 Stanford Alpaca 的标准 Prompt Template，将三元组转换为模型输入。

核心要点：
- 有 input 和无 input 使用不同模板
- `### Response:` 之后的部分是模型需要生成的内容
- Loss 只计算在 Response 部分

In [ ]:
PROMPT_WITH_INPUT = (
    "Below is an instruction that describes a task, "
    "paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)

PROMPT_WITHOUT_INPUT = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Response:\n"
)


def format_alpaca_prompt(item: Dict) -> Tuple[str, str]:
    """将 Alpaca 格式数据转换为 (prompt, full_text)。"""
    if item.get("input", "").strip():
        prompt = PROMPT_WITH_INPUT.format(
            instruction=item["instruction"],
            input=item["input"]
        )
    else:
        prompt = PROMPT_WITHOUT_INPUT.format(
            instruction=item["instruction"]
        )
    full_text = prompt + item["output"]
    return prompt, full_text


# 展示两个样例
for i in [0, 1]:
    prompt, full = format_alpaca_prompt(SFT_DATA[i])
    print(f"===== 样例 {i+1} (有input={bool(SFT_DATA[i]['input'])}) =====")
    print(full)
    print(f"\n--- Prompt 长度: {len(prompt)} chars, 全文长度: {len(full)} chars ---")
    print()

### 1.3 字符级 Tokenizer

为了在没有额外依赖的情况下完成实验，使用字符级 tokenizer。

真实场景中应使用 SentencePiece (LLaMA) 或 BPE (GPT) tokenizer。

In [ ]:
class CharTokenizer:
    """字符级 Tokenizer，用于教学演示。"""

    def __init__(self):
        # ASCII 可打印字符 + 常见特殊字符
        chars = list(set(
            'abcdefghijklmnopqrstuvwxyz'
            'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
            '0123456789'
            ' .,;:!?\'\"()-/\\@#$%^&*+=[]{}|<>~`'
            '\n\t'
        ))
        # 添加特殊 token
        special_tokens = ['<pad>', '<unk>', '<bos>', '<eos>']
        self.vocab = special_tokens + sorted(chars)
        self.stoi = {ch: i for i, ch in enumerate(self.vocab)}
        self.itos = {i: ch for i, ch in enumerate(self.vocab)}
        self.pad_id = self.stoi['<pad>']
        self.unk_id = self.stoi['<unk>']
        self.bos_id = self.stoi['<bos>']
        self.eos_id = self.stoi['<eos>']

    @property
    def vocab_size(self):
        return len(self.vocab)

    def encode(self, text: str) -> List[int]:
        return [self.stoi.get(c, self.unk_id) for c in text]

    def decode(self, ids: List[int]) -> str:
        return ''.join(self.itos.get(i, '?') for i in ids
                       if i not in (self.pad_id, self.bos_id, self.eos_id))


tokenizer = CharTokenizer()
print(f"Vocab size: {tokenizer.vocab_size}")

test_text = "Hello, World!\nThis is a test."
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)
print(f"Original:  '{test_text}'")
print(f"Encoded:   {encoded[:20]}... (len={len(encoded)})")
print(f"Decoded:   '{decoded}'")
print(f"Roundtrip: {test_text == decoded}")

### 1.4 SFT Dataset — 核心：Loss Mask 实现

SFT 的关键工程细节：**只在 output 部分计算 loss**。

```
tokens:  [Below, is, an, instruction, ..., ### Response:\n, The, capital, is, Paris, .]
labels:  [-100, -100, -100, -100, ..., -100,              The, capital, is, Paris, .]
          ←── prompt 部分: 不计算 loss ──→                 ←── output: 计算 loss ──→
```

In [ ]:
class AlpacaSFTDataset(Dataset):
    """
    Alpaca 格式的 SFT 数据集。
    
    核心功能：
    1. 将 (instruction, input, output) 转换为模型输入
    2. 构造 Loss Mask：prompt 部分标记为 -100，只在 output 部分计算 loss
    3. Padding 到统一长度
    """

    def __init__(self, data: List[Dict], tokenizer, max_len: int = 512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.processed = self._process_all()

    def _process_all(self):
        processed = []
        for item in self.data:
            prompt, full_text = format_alpaca_prompt(item)

            prompt_ids = self.tokenizer.encode(prompt)
            full_ids = self.tokenizer.encode(full_text)

            # 截断到 max_len
            full_ids = full_ids[:self.max_len]

            # 构造 labels: prompt 部分为 -100
            prompt_len = min(len(prompt_ids), len(full_ids))
            labels = [-100] * prompt_len + full_ids[prompt_len:]

            assert len(full_ids) == len(labels)
            processed.append({
                'input_ids': full_ids,
                'labels': labels,
                'prompt_len': prompt_len,
            })
        return processed

    def __len__(self):
        return len(self.processed)

    def __getitem__(self, idx):
        item = self.processed[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'labels': torch.tensor(item['labels'], dtype=torch.long),
        }


def collate_fn(batch):
    """动态 padding 到 batch 内最长序列。"""
    max_len = max(len(item['input_ids']) for item in batch)

    input_ids = []
    labels = []
    attention_mask = []

    for item in batch:
        seq_len = len(item['input_ids'])
        pad_len = max_len - seq_len

        input_ids.append(F.pad(item['input_ids'], (0, pad_len), value=0))
        labels.append(F.pad(item['labels'], (0, pad_len), value=-100))
        attention_mask.append(
            torch.cat([torch.ones(seq_len), torch.zeros(pad_len)])
        )

    return {
        'input_ids': torch.stack(input_ids),
        'labels': torch.stack(labels),
        'attention_mask': torch.stack(attention_mask),
    }


# 创建数据集
dataset = AlpacaSFTDataset(SFT_DATA, tokenizer, max_len=512)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

print(f"数据集大小: {len(dataset)}")

# 验证 Loss Mask
sample = dataset[0]
print(f"\n===== 验证 Loss Mask =====")
print(f"input_ids 长度: {len(sample['input_ids'])}")
print(f"labels 长度: {len(sample['labels'])}")

prompt_tokens = sum(1 for l in sample['labels'] if l == -100)
output_tokens = sum(1 for l in sample['labels'] if l != -100)
print(f"Prompt tokens (masked): {prompt_tokens}")
print(f"Output tokens (loss计算): {output_tokens}")

# 解码验证
print(f"\nPrompt 部分:")
print(tokenizer.decode(sample['input_ids'][:prompt_tokens].tolist())[:200] + "...")
print(f"\nOutput 部分:")
print(tokenizer.decode(sample['input_ids'][prompt_tokens:].tolist()))

## Part 2：模型构建

使用 Week 4 手写的 LLaMA 模型，缩小配置以适配单卡训练。

包含完整的 LLaMA 架构：RMSNorm + RoPE + SwiGLU + MHA。

In [ ]:
@dataclass
class LLaMAConfig:
    vocab_size: int = 100
    d_model: int = 256
    n_layers: int = 6
    n_heads: int = 8
    d_ff: int = 688      # round(2/3 * 4 * 256 / 8) * 8
    max_seq_len: int = 512
    dropout: float = 0.1
    norm_eps: float = 1e-6


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight


def precompute_freqs_cis(dim: int, max_seq_len: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis


def apply_rotary_emb(xq, xk, freqs_cis):
    xq_complex = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_complex = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))

    freqs_cis = freqs_cis[:xq.shape[1]].unsqueeze(0).unsqueeze(2)

    xq_out = torch.view_as_real(xq_complex * freqs_cis).flatten(-2)
    xk_out = torch.view_as_real(xk_complex * freqs_cis).flatten(-2)
    return xq_out.type_as(xq), xk_out.type_as(xk)


class Attention(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.d_head = config.d_model // config.n_heads

        self.wq = nn.Linear(config.d_model, config.d_model, bias=False)
        self.wk = nn.Linear(config.d_model, config.d_model, bias=False)
        self.wv = nn.Linear(config.d_model, config.d_model, bias=False)
        self.wo = nn.Linear(config.d_model, config.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, freqs_cis, mask=None):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q_r = q.transpose(1, 2)
        k_r = k.transpose(1, 2)
        q_r, k_r = apply_rotary_emb(q_r, k_r, freqs_cis)
        q = q_r.transpose(1, 2)
        k = k_r.transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.wo(out)


class SwiGLU_FFN(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.w_gate = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.w_up = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.w_down = nn.Linear(config.d_ff, config.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(
            self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))
        )


class LLaMABlock(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.attention_norm = RMSNorm(config.d_model, config.norm_eps)
        self.attention = Attention(config)
        self.ffn_norm = RMSNorm(config.d_model, config.norm_eps)
        self.feed_forward = SwiGLU_FFN(config)

    def forward(self, x, freqs_cis, mask=None):
        x = x + self.attention(self.attention_norm(x), freqs_cis, mask)
        x = x + self.feed_forward(self.ffn_norm(x))
        return x


class LLaMA(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.layers = nn.ModuleList(
            [LLaMABlock(config) for _ in range(config.n_layers)]
        )
        self.norm = RMSNorm(config.d_model, config.norm_eps)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.dropout = nn.Dropout(config.dropout)

        self.freqs_cis = precompute_freqs_cis(
            config.d_model // config.n_heads, config.max_seq_len
        )

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.dropout(self.tok_emb(input_ids))

        freqs_cis = self.freqs_cis[:T].to(x.device)
        mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)

        for layer in self.layers:
            x = layer(x, freqs_cis, mask)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return logits, loss


# 创建模型
config = LLaMAConfig(vocab_size=tokenizer.vocab_size)
model = LLaMA(config).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {total_params:,} ({total_params/1e6:.2f}M)")
print(f"配置: d_model={config.d_model}, n_layers={config.n_layers}, "
      f"n_heads={config.n_heads}, d_ff={config.d_ff}")

## Part 3：SFT 训练

### 3.1 训练配置

SFT 的训练配置与预训练有关键区别：
- **学习率更低**：$2 \times 10^{-4}$（预训练通常 $3 \times 10^{-4}$）
- **Epoch 更少**：3-5 epochs（避免过拟合到微调数据）
- **Loss 只计算在 output 部分**（通过 labels 中的 -100 实现）

In [ ]:
# 训练超参数
NUM_EPOCHS = 30       # 数据量小，多训几轮
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 20
GRAD_CLIP = 1.0

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),
)

total_steps = NUM_EPOCHS * len(dataloader)
print(f"总训练步数: {total_steps}")
print(f"每 epoch 步数: {len(dataloader)}")


def get_lr(step, warmup_steps, total_steps, max_lr, min_lr=1e-6):
    """Cosine decay with linear warmup."""
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

### 3.2 训练前的模型行为（基线）

在训练前观察模型的输出——未经 SFT 的模型应该只会「续写」而非「遵循指令」。

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt: str, max_new_tokens: int = 150,
             temperature: float = 0.8, top_k: int = 50):
    """自回归文本生成。"""
    model.eval()
    input_ids = torch.tensor(
        [tokenizer.encode(prompt)], dtype=torch.long, device=device
    )

    for _ in range(max_new_tokens):
        input_truncated = input_ids[:, -model.config.max_seq_len:]
        logits, _ = model(input_truncated)
        logits = logits[:, -1, :] / temperature

        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_id], dim=1)

        if next_id.item() == tokenizer.eos_id:
            break

    model.train()
    return tokenizer.decode(input_ids[0].tolist())


# 训练前测试
test_prompts = [
    format_alpaca_prompt({
        "instruction": "What is machine learning?",
        "input": "", "output": ""
    })[0],
    format_alpaca_prompt({
        "instruction": "Write a short poem about the moon.",
        "input": "", "output": ""
    })[0],
]

print("===== 训练前的模型输出（随机权重）=====")
for i, prompt in enumerate(test_prompts):
    output = generate(model, tokenizer, prompt, max_new_tokens=100)
    print(f"\n--- 测试 {i+1} ---")
    print(output[:300])
    print("...")

### 3.3 训练循环

核心区别于预训练：Loss 只在 output 部分计算（通过 `ignore_index=-100` 实现）。

In [ ]:
def train_sft(model, dataloader, optimizer, num_epochs, warmup_steps,
              grad_clip, total_steps):
    """SFT 训练循环。"""
    model.train()
    train_losses = []
    global_step = 0

    for epoch in range(num_epochs):
        epoch_loss = 0
        num_batches = 0

        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            # 学习率调度
            lr = get_lr(global_step, warmup_steps, total_steps, LEARNING_RATE)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            # 前向传播（Loss Mask 通过 labels 中的 -100 自动生效）
            logits, loss = model(input_ids, labels)

            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            epoch_loss += loss.item()
            num_batches += 1
            global_step += 1
            train_losses.append(loss.item())

        avg_loss = epoch_loss / num_batches
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{num_epochs} | "
                  f"Loss: {avg_loss:.4f} | LR: {lr:.2e}")

    return train_losses


print("开始 SFT 训练...\n")
start_time = time.time()

train_losses = train_sft(
    model, dataloader, optimizer,
    num_epochs=NUM_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    grad_clip=GRAD_CLIP,
    total_steps=total_steps,
)

elapsed = time.time() - start_time
print(f"\n训练完成! 耗时: {elapsed:.1f}s")

In [ ]:
# 绘制 Loss 曲线
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 原始 loss
ax1.plot(train_losses, alpha=0.3, color='blue', label='Step Loss')
# 滑动平均
window = min(20, len(train_losses) // 5)
if window > 1:
    smoothed = np.convolve(train_losses, np.ones(window)/window, mode='valid')
    ax1.plot(range(window-1, len(train_losses)), smoothed,
             color='red', linewidth=2, label=f'Smoothed (window={window})')
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('SFT Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 每 epoch 平均 loss
steps_per_epoch = len(dataloader)
epoch_losses = []
for i in range(0, len(train_losses), steps_per_epoch):
    epoch_batch = train_losses[i:i+steps_per_epoch]
    if epoch_batch:
        epoch_losses.append(np.mean(epoch_batch))

ax2.plot(range(1, len(epoch_losses)+1), epoch_losses, 'o-', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Average Loss')
ax2.set_title('SFT Loss per Epoch')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"初始 Loss: {train_losses[0]:.4f}")
print(f"最终 Loss: {train_losses[-1]:.4f}")
print(f"Loss 下降: {train_losses[0] - train_losses[-1]:.4f}")

## Part 4：生成与评估

### 4.1 训练后的模型行为

观察 SFT 后模型的输出变化——模型应该从「随机续写」转变为「遵循指令」。

In [ ]:
# 测试训练集中的指令（模型应该能较好地复现）
print("===== 训练后的模型输出（训练集内指令）=====")
for i, prompt in enumerate(test_prompts):
    output = generate(model, tokenizer, prompt, max_new_tokens=200,
                      temperature=0.7)
    # 只显示 Response 部分
    resp_marker = "### Response:\n"
    if resp_marker in output:
        response = output.split(resp_marker, 1)[1]
    else:
        response = output
    print(f"\n--- 测试 {i+1} ---")
    print(f"Response: {response[:300]}")
    print()

In [ ]:
# 测试训练集外的新指令（泛化能力）
unseen_prompts = [
    format_alpaca_prompt({
        "instruction": "What is the largest planet in our solar system?",
        "input": "", "output": ""
    })[0],
    format_alpaca_prompt({
        "instruction": "Write a short poem about the sun.",
        "input": "", "output": ""
    })[0],
    format_alpaca_prompt({
        "instruction": "Summarize the following text in one sentence.",
        "input": "Deep learning has transformed the field of computer vision, enabling machines to recognize objects and scenes with superhuman accuracy.",
        "output": ""
    })[0],
]

print("===== 训练后的模型输出（未见过的指令）=====")
for i, prompt in enumerate(unseen_prompts):
    output = generate(model, tokenizer, prompt, max_new_tokens=200,
                      temperature=0.7)
    resp_marker = "### Response:\n"
    if resp_marker in output:
        response = output.split(resp_marker, 1)[1]
    else:
        response = output
    print(f"\n--- 新指令 {i+1} ---")
    print(f"Response: {response[:300]}")
    print()

### 4.2 行为变化分析

```
训练前（随机权重）:
  模型输出随机字符，无任何语义
  → 因为模型既没有预训练知识，也没有指令遵循能力

训练后（SFT）:
  对训练集内的指令：应该能生成较好的回答（记住了训练数据）
  对新指令：泛化能力有限（数据量太少，且模型没有预训练基础）

在真实场景中:
  预训练模型 + SFT → 即使只有少量 SFT 数据也能有很好的泛化
  因为预训练已经给了模型知识，SFT 只需教会它「回答格式」
  → 回顾 Day 1 的「表面对齐假说」
```

## Part 5：Adapter 微调对比

在同一个模型上对比 Full Fine-tuning 和 Adapter 微调的效果差异。

### 5.1 实现 Adapter

In [ ]:
class Adapter(nn.Module):
    """瓶颈 Adapter 模块。"""

    def __init__(self, d_model: int, bottleneck_dim: int):
        super().__init__()
        self.down = nn.Linear(d_model, bottleneck_dim, bias=True)
        self.up = nn.Linear(bottleneck_dim, d_model, bias=True)
        self.act = nn.ReLU()

        # 近零初始化
        nn.init.normal_(self.down.weight, std=0.01)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))


class LLaMABlockWithAdapter(nn.Module):
    """带 Adapter 的 LLaMA Block。冻结原始权重，只训练 Adapter。"""

    def __init__(self, original_block: LLaMABlock, d_model: int, bottleneck_dim: int):
        super().__init__()
        self.block = original_block
        self.adapter_attn = Adapter(d_model, bottleneck_dim)
        self.adapter_ffn = Adapter(d_model, bottleneck_dim)

    def forward(self, x, freqs_cis, mask=None):
        # Attention + Adapter
        attn_out = self.block.attention(
            self.block.attention_norm(x), freqs_cis, mask
        )
        x = x + self.adapter_attn(attn_out)

        # FFN + Adapter
        ffn_out = self.block.feed_forward(self.block.ffn_norm(x))
        x = x + self.adapter_ffn(ffn_out)

        return x


class LLaMAWithAdapter(nn.Module):
    """带 Adapter 的 LLaMA 模型。冻结原始参数，只训练 Adapter。"""

    def __init__(self, base_model: LLaMA, bottleneck_dim: int = 32):
        super().__init__()
        self.config = base_model.config
        self.tok_emb = base_model.tok_emb
        self.norm = base_model.norm
        self.lm_head = base_model.lm_head
        self.dropout = base_model.dropout
        self.freqs_cis = base_model.freqs_cis

        # 用 Adapter 包装每个 Block
        self.layers = nn.ModuleList([
            LLaMABlockWithAdapter(block, self.config.d_model, bottleneck_dim)
            for block in base_model.layers
        ])

        # 冻结所有非 Adapter 参数
        for name, param in self.named_parameters():
            if 'adapter' not in name:
                param.requires_grad = False

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.dropout(self.tok_emb(input_ids))
        freqs_cis = self.freqs_cis[:T].to(x.device)
        mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)

        for layer in self.layers:
            x = layer(x, freqs_cis, mask)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return logits, loss


# 创建 Adapter 模型（基于新的随机初始化基座）
base_model_adapter = LLaMA(config).to(device)
adapter_model = LLaMAWithAdapter(base_model_adapter, bottleneck_dim=32).to(device)

total_params = sum(p.numel() for p in adapter_model.parameters())
trainable_params = sum(p.numel() for p in adapter_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Adapter 模型参数统计:")
print(f"  总参数量: {total_params:,}")
print(f"  可训练参数 (Adapter): {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
print(f"  冻结参数: {frozen_params:,} ({frozen_params/total_params*100:.2f}%)")

In [ ]:
# 训练 Adapter 模型
adapter_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, adapter_model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),
)

print("开始 Adapter SFT 训练...\n")
start_time = time.time()

adapter_model.train()
adapter_losses = []
global_step = 0

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    num_batches = 0

    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        lr = get_lr(global_step, WARMUP_STEPS, total_steps, LEARNING_RATE)
        for pg in adapter_optimizer.param_groups:
            pg['lr'] = lr

        logits, loss = adapter_model(input_ids, labels)

        adapter_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, adapter_model.parameters()),
            GRAD_CLIP
        )
        adapter_optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1
        global_step += 1
        adapter_losses.append(loss.item())

    avg_loss = epoch_loss / num_batches
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | LR: {lr:.2e}")

elapsed = time.time() - start_time
print(f"\n训练完成! 耗时: {elapsed:.1f}s")

In [ ]:
# 对比 Full FT vs Adapter 的 Loss 曲线
fig, ax = plt.subplots(figsize=(12, 5))

# 滑动平均
window = min(20, len(train_losses) // 5)
if window > 1:
    smooth_full = np.convolve(train_losses, np.ones(window)/window, mode='valid')
    smooth_adapter = np.convolve(adapter_losses, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(train_losses)), smooth_full,
            color='blue', linewidth=2, label='Full Fine-tuning')
    ax.plot(range(window-1, len(adapter_losses)), smooth_adapter,
            color='red', linewidth=2, label='Adapter')
else:
    ax.plot(train_losses, label='Full Fine-tuning', alpha=0.7)
    ax.plot(adapter_losses, label='Adapter', alpha=0.7)

ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Full Fine-tuning vs Adapter: SFT Loss Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFull FT — 最终 Loss: {train_losses[-1]:.4f}, 可训练参数: {total_params:,} (100%)")
print(f"Adapter — 最终 Loss: {adapter_losses[-1]:.4f}, 可训练参数: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")

## Part 6：实验总结

### 6.1 关键发现

```
1. Loss Mask 的作用:
   只在 output 部分计算 loss → 模型学习「给出好的回答」而非「重复问题」
   这是 SFT 区别于普通预训练的核心工程细节

2. 数据量的影响:
   20 条数据 → 模型能记住训练数据，但泛化能力有限
   真实场景需要 1K~100K 条多样化数据

3. 预训练基础的重要性:
   本实验从随机初始化开始 → 泛化能力差
   在预训练模型上做 SFT → 泛化能力强（表面对齐假说）

4. Full FT vs Adapter:
   Full FT 更新所有参数 → 学习能力更强
   Adapter 只更新 ~2% 参数 → 参数效率高
   在小数据集上两者差距有限
   真实场景中差距更为显著（预训练基座 + 大量 SFT 数据）
```

### 6.2 与真实 Alpaca 训练的差异

| 维度 | 本实验 | 真实 Alpaca |
|------|--------|------------|
| 基座模型 | 随机初始化 (~2M) | LLaMA-7B (预训练) |
| Tokenizer | 字符级 | SentencePiece BPE (32K) |
| 数据量 | 20 条 | 52,000 条 |
| 训练 Epoch | 30 | 3 |
| 硬件 | 单 GPU | 4×A100 80GB |
| 泛化能力 | 极弱 | 接近 text-davinci-003 |

### 6.3 自检题

1. **SFT 的 Loss 为什么只计算在 output 部分？** 如果在 instruction 部分也计算会怎样？
2. **`ignore_index=-100` 在 CrossEntropyLoss 中的作用是什么？**
3. **为什么训练后的模型对未见过的指令泛化能力差？** 如果有预训练基础会怎样？
4. **Adapter 的近零初始化为什么重要？** 观察训练初期 Adapter 模型的行为。
5. **对比 Full FT 和 Adapter 的训练速度差异。** Adapter 的梯度计算量为什么更少？
6. **在真实场景中，如何用 Day 3 的 Self-Instruct 管线生成 SFT 数据？**

### 6.4 产出清单

- [ ] 完成 Alpaca Prompt Template 的实现（有 input / 无 input 两种）
- [ ] 实现 Loss Mask（labels 中 prompt 部分标记为 -100）
- [ ] 完成完整的 SFT 训练循环（含 Cosine LR + Gradient Clipping）
- [ ] 观察训练前后模型输出的行为变化
- [ ] 实现 Adapter 并对比 Full FT vs Adapter 的训练曲线
- [ ] 分析本实验与真实 Alpaca 训练的差异